In [ ]:
import re
import json
import math
import time
import random
from collections import Counter

import numpy as np
import pandas as pd
import scipy.sparse as sp

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


In [ ]:
TRAIN_PATH = "/content/train.csv"
VAL_PATH   = "/content/validation.csv"
TEST_PATH  = "/content/test.csv"

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

# Basic cleanup: drop empty / NaN rows just in case
for d in (train_df, val_df, test_df):
    d.dropna(subset=["text", "label"], inplace=True)
    d["text"] = d["text"].astype(str)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

languages = sorted(train_df["label"].unique())
print(f"\nNumber of languages: {len(languages)}")
print(languages)

label2idx = {lang: i for i, lang in enumerate(languages)}
idx2label = {i: lang for lang, i in label2idx.items()}


Train: (18400, 2)
Validation: (2300, 2)
Test: (2300, 2)

Number of languages: 23
['asm_Beng', 'ben_Beng', 'brx_Deva', 'doi_Deva', 'gom_Deva', 'guj_Gujr', 'hin_Deva', 'kan_Knda', 'kas_Arab', 'khasi', 'mai_Deva', 'mal_Mlym', 'mar_Deva', 'mni_Mtei', 'npi_Deva', 'ory_Orya', 'pan_Guru', 'san_Deva', 'santhali', 'snd_Deva', 'tam_Taml', 'tel_Telu', 'urd_Arab']


In [ ]:
print("TRAIN label distribution")
print(train_df["label"].value_counts().sort_index())
print("\nVALIDATION label distribution")
print(val_df["label"].value_counts().sort_index())
print("\nTEST label distribution")
print(test_df["label"].value_counts().sort_index())


TRAIN label distribution
label
asm_Beng    800
ben_Beng    800
brx_Deva    800
doi_Deva    800
gom_Deva    800
guj_Gujr    800
hin_Deva    800
kan_Knda    800
kas_Arab    800
khasi       800
mai_Deva    800
mal_Mlym    800
mar_Deva    800
mni_Mtei    800
npi_Deva    800
ory_Orya    800
pan_Guru    800
san_Deva    800
santhali    800
snd_Deva    800
tam_Taml    800
tel_Telu    800
urd_Arab    800
Name: count, dtype: int64

VALIDATION label distribution
label
asm_Beng    100
ben_Beng    100
brx_Deva    100
doi_Deva    100
gom_Deva    100
guj_Gujr    100
hin_Deva    100
kan_Knda    100
kas_Arab    100
khasi       100
mai_Deva    100
mal_Mlym    100
mar_Deva    100
mni_Mtei    100
npi_Deva    100
ory_Orya    100
pan_Guru    100
san_Deva    100
santhali    100
snd_Deva    100
tam_Taml    100
tel_Telu    100
urd_Arab    100
Name: count, dtype: int64

TEST label distribution
label
asm_Beng    100
ben_Beng    100
brx_Deva    100
doi_Deva    100
gom_Deva    100
guj_Gujr    100
hin_Deva    100
k

In [ ]:
def word_tokenize(text):
    """Whitespace tokenizer (Indic scripts don't use spaces between characters the way
    Latin scripts do, but IndicCorp sentences are space-separated at the word level)."""
    text = text.strip()
    return re.findall(r"\S+", text)


def word_ngrams(tokens, n):
    if len(tokens) < n:
        return []
    return [" ".join(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]


def char_ngrams(text, n):
    text = re.sub(r"\s+", " ", text).strip()
    if len(text) < n:
        return []
    return [text[i:i + n] for i in range(len(text) - n + 1)]


def get_features(row):
    """Build the full feature list (word unigrams+bigrams, char 2/3/4-grams) for one sample.

    `row` can be a pandas Series / dict with a "text" field, or a raw string.
    """
    text = row["text"] if not isinstance(row, str) else row
    tokens = word_tokenize(text)

    features = []
    features += ["W1_" + f for f in word_ngrams(tokens, 1)]
    features += ["W2_" + f for f in word_ngrams(tokens, 2)]
    features += ["C2_" + f for f in char_ngrams(text, 2)]
    features += ["C3_" + f for f in char_ngrams(text, 3)]
    features += ["C4_" + f for f in char_ngrams(text, 4)]
    return features


# quick sanity check
demo_feats = get_features(train_df.iloc[0])
print("Sentence:", train_df.iloc[0]["text"])
print("Language:", train_df.iloc[0]["label"])
print("Number of features extracted:", len(demo_feats))
print("Sample features:", demo_feats[:10])


Sentence: বন্তি প্ৰজ্বলন কৰে অৱসৰপ্ৰাপ্ত শিক্ষক ললিত সন্দিকৈয়ে৷ সভাৰ আৰম্ভণি হয় বিদ্যালয়ৰ ছাত্ৰী প্ৰাচীপ্ৰিয়া দত্ত, পৰিশ্মিতা দাস আৰু কবিতা শইকীয়াৰ দ্বাৰা পৰিবেশিত ভূপেন হাজৰিকাৰ গীতেৰে৷ বিদ্যালয়ৰ প্ৰধান শিক্ষয়িত্ৰী পাৰুল দেৱীয়ে আঁত ধৰা সম্বৰ্ধনা অনুষ্ঠানত উপস্থিত আমন্ত্ৰিতসকলক সম্বৰ্ধনা জনোৱা হয়।
Language: asm_Beng
Number of features extracted: 933
Sample features: ['W1_বন্তি', 'W1_প্ৰজ্বলন', 'W1_কৰে', 'W1_অৱসৰপ্ৰাপ্ত', 'W1_শিক্ষক', 'W1_ললিত', 'W1_সন্দিকৈয়ে৷', 'W1_সভাৰ', 'W1_আৰম্ভণি', 'W1_হয়']


In [ ]:
def build_vocabulary(texts, min_df=2, max_features=60000):
    """Builds vocabulary {feature: index} and the idf array, from a list of raw texts."""
    N = len(texts)
    doc_freq = Counter()

    for text in texts:
        feats = set(get_features(text))   # set() -> count each feature once per doc
        doc_freq.update(feats)

    # filter by min_df
    filtered = [(f, c) for f, c in doc_freq.items() if c >= min_df]
    # keep the most globally frequent (by document frequency) features, capped at max_features
    filtered.sort(key=lambda x: x[1], reverse=True)
    filtered = filtered[:max_features]

    vocabulary = {feat: i for i, (feat, _) in enumerate(filtered)}
    doc_freq_arr = np.zeros(len(vocabulary), dtype=np.float64)
    for feat, c in filtered:
        doc_freq_arr[vocabulary[feat]] = c

    idf = np.log((1.0 + N) / (1.0 + doc_freq_arr)) + 1.0
    return vocabulary, idf, N


t0 = time.time()
vocabulary, idf, N_train = build_vocabulary(
    train_df["text"].tolist(),
    min_df=2,
    max_features=60000,
)
print(f"Vocabulary built in {time.time() - t0:.1f}s")
print("Vocabulary size:", len(vocabulary))


Vocabulary built in 5.4s
Vocabulary size: 60000


In [ ]:
def calculate_tfidf(features, vocabulary, idf, normalize=True):
    """Turns a list of raw feature strings into a sparse TF-IDF vector
    represented as {vocab_index: tfidf_value}.

    - term frequency (tf) is the count of the n-gram in the document,
      normalized by the total number of n-gram tokens in the document.
    - tfidf = tf * idf
    - the resulting vector is L2-normalized (standard practice, makes documents
      of different lengths comparable and stabilizes logistic-regression training).
    """
    if len(features) == 0:
        return {}

    counts = Counter(f for f in features if f in vocabulary)
    total_tokens = len(features)

    vector = {}
    for feat, count in counts.items():
        idx = vocabulary[feat]
        tf = count / total_tokens
        vector[idx] = tf * idf[idx]

    if normalize and vector:
        norm = math.sqrt(sum(v * v for v in vector.values()))
        if norm > 0:
            for k in vector:
                vector[k] /= norm

    return vector


### Demo — TF-IDF vector for a single sentence

(matches the check requested in the assignment)

In [ ]:
features = get_features(train_df.iloc[0])

vector = calculate_tfidf(
    features,
    vocabulary,
    idf
)

print("Number of non-zero features:",
      len(vector))

print("\nFirst 20 TF-IDF values:")

for index, value in list(vector.items())[:20]:
    print(index, ":", value)


Number of non-zero features: 452

First 20 TF-IDF values:
29326 : 0.03694807938119424
26112 : 0.03641453401047904
2497 : 0.027128427716312416
33166 : 0.03754711166624118
16395 : 0.03437759993183169
5683 : 0.030081606150657383
251 : 0.06405062959129071
256 : 0.08558697336856719
395 : 0.022377216005246187
184 : 0.041334000918906245
108 : 0.11629198631506144
319 : 0.13122603739211824
647 : 0.18749307311976438
5611 : 0.030035728083082733
709 : 0.09446957511394401
2340 : 0.026906859252028875
13277 : 0.03346830823230286
197 : 0.06234734812841688
95 : 0.037857989480114596
593 : 0.023250921398168246


In [ ]:
def vectorize_dataset(df, vocabulary, idf, verbose_every=5000):
    n = len(df)
    rows_idx, cols_idx, data = [], [], []

    texts = df["text"].tolist()
    for r, text in enumerate(texts):
        feats = get_features(text)
        vec = calculate_tfidf(feats, vocabulary, idf)
        for c, v in vec.items():
            rows_idx.append(r)
            cols_idx.append(c)
            data.append(v)
        if verbose_every and (r + 1) % verbose_every == 0:
            print(f"  vectorized {r + 1}/{n}")

    mat = sp.csr_matrix((data, (rows_idx, cols_idx)), shape=(n, len(vocabulary)))
    return mat


print("Vectorizing train set...")
t0 = time.time()
X_train = vectorize_dataset(train_df, vocabulary, idf)
y_train = np.array([label2idx[l] for l in train_df["label"]])
print(f"  done in {time.time() - t0:.1f}s -> {X_train.shape}")

print("Vectorizing validation set...")
X_val = vectorize_dataset(val_df, vocabulary, idf)
y_val = np.array([label2idx[l] for l in val_df["label"]])
print(f"  -> {X_val.shape}")

print("Vectorizing test set...")
X_test = vectorize_dataset(test_df, vocabulary, idf)
y_test = np.array([label2idx[l] for l in test_df["label"]])
print(f"  -> {X_test.shape}")


Vectorizing train set...
  vectorized 5000/18400
  vectorized 10000/18400
  vectorized 15000/18400
  done in 10.6s -> (18400, 60000)
Vectorizing validation set...
  -> (2300, 60000)
Vectorizing test set...
  -> (2300, 60000)


## 5. Logistic Regression from scratch (multinomial / softmax regression)

We implement **multinomial logistic regression** (a direct generalization of binary logistic regression to `K` classes using the softmax function) trained with **mini-batch gradient descent**. No `sklearn.linear_model` class is used — the forward pass, softmax, cross-entropy loss, gradients, and the parameter-update rule are all written by hand with NumPy / SciPy sparse matrix operations (SciPy sparse matrices are just used as an efficient container for our own TF-IDF numbers — not a modeling shortcut).

**Model**

$$z = XW^\top + b, \qquad p = \text{softmax}(z), \qquad \hat{y} = \arg\max_k p_k$$

**Loss** (cross-entropy + L2 regularization):

$$\mathcal{L} = -\frac{1}{n}\sum_{i=1}^{n} \log p_{i,y_i} \; + \; \frac{\lambda}{2}\lVert W \rVert_2^2$$

**Gradients** (per mini-batch of size `m`):

$$\frac{\partial \mathcal{L}}{\partial W} = \frac{1}{m}X^\top (P - Y_{\text{onehot}}) \; + \; \lambda W, \qquad \frac{\partial \mathcal{L}}{\partial b} = \frac{1}{m}\sum_i (P - Y_{\text{onehot}})_i$$

In [ ]:
class SoftmaxRegressionScratch:
    """Multinomial Logistic Regression trained with mini-batch gradient descent.
    Implemented from scratch with NumPy / SciPy sparse ops only (no sklearn).
    """

    def __init__(self, num_features, num_classes, learning_rate=0.5,
                 epochs=25, batch_size=64, l2_reg=1e-4, seed=42):
        rng = np.random.default_rng(seed)
        self.W = rng.normal(0, 0.01, size=(num_classes, num_features))
        self.b = np.zeros(num_classes)
        self.lr = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size
        self.l2 = l2_reg
        self.num_classes = num_classes
        self.history = {"train_loss": [], "val_acc": []}

    @staticmethod
    def softmax(Z):
        Z = Z - Z.max(axis=1, keepdims=True)   # numerical stability
        E = np.exp(Z)
        return E / E.sum(axis=1, keepdims=True)

    def _forward(self, X):
        Z = X.dot(self.W.T) + self.b   # sparse (n x d) dot dense (d x k) -> dense (n x k)
        return self.softmax(np.asarray(Z))

    def fit(self, X, y, X_val=None, y_val=None, verbose=True):
        n = X.shape[0]
        Y = np.zeros((n, self.num_classes))
        Y[np.arange(n), y] = 1.0
        indices = np.arange(n)

        for epoch in range(self.epochs):
            rng = np.random.default_rng(1000 + epoch)
            rng.shuffle(indices)
            total_loss = 0.0

            for start in range(0, n, self.batch_size):
                batch_idx = indices[start:start + self.batch_size]
                Xb = X[batch_idx]
                Yb = Y[batch_idx]
                yb = y[batch_idx]
                bs = Xb.shape[0]

                Zb = Xb.dot(self.W.T) + self.b
                Pb = self.softmax(np.asarray(Zb))

                dZ = (Pb - Yb) / bs                      # (bs x k)
                grad_W = np.asarray(Xb.T.dot(dZ)).T       # (k x d)
                grad_W += self.l2 * self.W
                grad_b = dZ.sum(axis=0)

                self.W -= self.lr * grad_W
                self.b -= self.lr * grad_b

                logp = np.log(np.clip(Pb[np.arange(bs), yb], 1e-12, None))
                total_loss += -logp.sum()

            avg_loss = total_loss / n
            self.history["train_loss"].append(avg_loss)

            msg = f"Epoch {epoch + 1:>3}/{self.epochs} - train_loss: {avg_loss:.4f}"
            if X_val is not None and y_val is not None:
                val_preds = self.predict(X_val)
                val_acc = float(np.mean(val_preds == y_val))
                self.history["val_acc"].append(val_acc)
                msg += f" - val_acc: {val_acc:.4f}"
            if verbose:
                print(msg)

    def predict_proba(self, X):
        return self._forward(X)

    def predict(self, X):
        P = self.predict_proba(X)
        return np.argmax(P, axis=1)


In [ ]:
clf = SoftmaxRegressionScratch(
    num_features=len(vocabulary),
    num_classes=len(languages),
    learning_rate=0.7,
    epochs=25,
    batch_size=64,
    l2_reg=1e-4,
    seed=RANDOM_SEED,
)

t0 = time.time()
clf.fit(X_train, y_train, X_val, y_val)
print(f"\nTraining finished in {time.time() - t0:.1f}s")


Epoch   1/25 - train_loss: 2.7206 - val_acc: 0.8943
Epoch   2/25 - train_loss: 2.0227 - val_acc: 0.9204
Epoch   3/25 - train_loss: 1.5695 - val_acc: 0.9139
Epoch   4/25 - train_loss: 1.2913 - val_acc: 0.9230
Epoch   5/25 - train_loss: 1.1112 - val_acc: 0.9165
Epoch   6/25 - train_loss: 0.9872 - val_acc: 0.9222
Epoch   7/25 - train_loss: 0.8963 - val_acc: 0.9304
Epoch   8/25 - train_loss: 0.8271 - val_acc: 0.9343
Epoch   9/25 - train_loss: 0.7727 - val_acc: 0.9343
Epoch  10/25 - train_loss: 0.7287 - val_acc: 0.9357
Epoch  11/25 - train_loss: 0.6924 - val_acc: 0.9317
Epoch  12/25 - train_loss: 0.6620 - val_acc: 0.9330
Epoch  13/25 - train_loss: 0.6363 - val_acc: 0.9343
Epoch  14/25 - train_loss: 0.6142 - val_acc: 0.9370
Epoch  15/25 - train_loss: 0.5950 - val_acc: 0.9339
Epoch  16/25 - train_loss: 0.5780 - val_acc: 0.9378
Epoch  17/25 - train_loss: 0.5634 - val_acc: 0.9374
Epoch  18/25 - train_loss: 0.5501 - val_acc: 0.9365
Epoch  19/25 - train_loss: 0.5384 - val_acc: 0.9348
Epoch  20/25

In [ ]:
def confusion_counts(y_true, y_pred, num_classes):
    """Returns TP, FP, FN, TN arrays (one entry per class)."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    tp = np.zeros(num_classes)
    fp = np.zeros(num_classes)
    fn = np.zeros(num_classes)
    tn = np.zeros(num_classes)
    for c in range(num_classes):
        tp[c] = np.sum((y_pred == c) & (y_true == c))
        fp[c] = np.sum((y_pred == c) & (y_true != c))
        fn[c] = np.sum((y_pred != c) & (y_true == c))
        tn[c] = np.sum((y_pred != c) & (y_true != c))
    return tp, fp, fn, tn


def precision_recall_f1_per_class(y_true, y_pred, num_classes):
    tp, fp, fn, _ = confusion_counts(y_true, y_pred, num_classes)
    precision = np.divide(tp, tp + fp, out=np.zeros_like(tp), where=(tp + fp) > 0)
    recall = np.divide(tp, tp + fn, out=np.zeros_like(tp), where=(tp + fn) > 0)
    denom = precision + recall
    f1 = np.divide(2 * precision * recall, denom, out=np.zeros_like(tp), where=denom > 0)
    return precision, recall, f1


def macro_f1_score(y_true, y_pred, num_classes):
    _, _, f1 = precision_recall_f1_per_class(y_true, y_pred, num_classes)
    return float(np.mean(f1))


def accuracy_score_custom(y_true, y_pred):
    return float(np.mean(np.asarray(y_true) == np.asarray(y_pred)))


def classification_report_custom(y_true, y_pred, num_classes, idx2label):
    precision, recall, f1 = precision_recall_f1_per_class(y_true, y_pred, num_classes)
    support = np.array([np.sum(np.asarray(y_true) == c) for c in range(num_classes)])
    rows = []
    for c in range(num_classes):
        rows.append({
            "language": idx2label[c],
            "precision": precision[c],
            "recall": recall[c],
            "f1": f1[c],
            "support": int(support[c]),
        })
    report_df = pd.DataFrame(rows).set_index("language")
    return report_df


In [ ]:
# ---- Validation set ----
val_preds = clf.predict(X_val)
val_acc = accuracy_score_custom(y_val, val_preds)
val_macro_f1 = macro_f1_score(y_val, val_preds, len(languages))

print(f"Validation accuracy : {val_acc:.4f}")
print(f"Validation macro-F1 : {val_macro_f1:.4f}")


Validation accuracy : 0.9387
Validation macro-F1 : 0.9376


In [ ]:
# ---- Test set ----
test_preds = clf.predict(X_test)
test_acc = accuracy_score_custom(y_test, test_preds)
test_macro_f1 = macro_f1_score(y_test, test_preds, len(languages))

print(f"Test accuracy : {test_acc:.4f}")
print(f"Test macro-F1 : {test_macro_f1:.4f}")

print("\nPer-language classification report (test set):")
report_df = classification_report_custom(y_test, test_preds, len(languages), idx2label)
report_df.sort_values("f1")


Test accuracy : 0.9291
Test macro-F1 : 0.9281

Per-language classification report (test set):


,precision,recall,f1,support
language,,,,
gom_Deva,0.698413,0.44,0.539877,100
mar_Deva,0.629630,0.85,0.723404,100
mai_Deva,0.825243,0.85,0.837438,100
doi_Deva,0.854369,0.88,0.866995,100
hin_Deva,0.798319,0.95,0.867580,100
brx_Deva,1.000000,0.77,0.870056,100
san_Deva,0.808333,0.97,0.881818,100
khasi,0.951456,0.98,0.965517,100
npi_Deva,0.989583,0.95,0.969388,100


In [ ]:
import pickle

artifacts = {
    "vocabulary": vocabulary,
    "idf": idf,
    "W": clf.W,
    "b": clf.b,
    "label2idx": label2idx,
    "idx2label": idx2label,
}

with open("language_id_model.pkl", "wb") as f:
    pickle.dump(artifacts, f)

print("Saved model artifacts to language_id_model.pkl")


Saved model artifacts to language_id_model.pkl


In [ ]:
def predict_language(text, vocabulary, idf, model, idx2label, top_k=3):
    feats = get_features({"text": text})
    vec = calculate_tfidf(feats, vocabulary, idf)
    if not vec:
        return []
    cols = list(vec.keys())
    data = list(vec.values())
    x = sp.csr_matrix((data, ([0] * len(cols), cols)), shape=(1, len(vocabulary)))
    probs = model.predict_proba(x)[0]
    top_idx = np.argsort(probs)[::-1][:top_k]
    return [(idx2label[i], float(probs[i])) for i in top_idx]


# Example (replace with any sentence from your test set)
sample_text = test_df.iloc[0]["text"]
print("Sentence:", sample_text)
print("True label:", test_df.iloc[0]["label"])
print("Predictions:", predict_language(sample_text, vocabulary, idf, clf, idx2label))


Sentence: ডিজিটেল সংবাদ, ওদালগুৰিঃ ‘অসমীয়া প্ৰতিদিন’ৰ ডিজিটেল সংস্কৰণত প্ৰকাশিত এটি বাতৰিৰ পাছতেই সোমবাৰে তাৎক্ষণিকভাৱে ওদালগুৰি জিলা উপায়ুক্ত প্ৰকাশ ৰঞ্জন ঘৰফলীয়াই প্ৰতিমা বিসৰ্জনস্থলী পৰিষ্কাৰৰ নিৰ্দেশ প্ৰদান কৰে ওদালগুৰি গড়কাপ্তানী বিভাগক।
True label: asm_Beng
Predictions: [('asm_Beng', 0.9397424425009095), ('ben_Beng', 0.03458135685535395), ('mni_Mtei', 0.0018877137956886149)]
